
# Telco Churn — Подготовка и EDA (00)

Мини-курс **«ML для аналитиков»** — итоговый проект (часть 1 из 2).

**Что делаем здесь:**
1) Загружаем датасет Telco Churn.
2) Чистим данные (особенно `TotalCharges`).
3) Делаем первичный анализ (EDA) и готовим фичи.
4) Сохраняем подготовленный датасет для следующего ноутбука.


In [ ]:
# Импорты
import pandas as pd
import numpy as np
import io, textwrap, os

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
print("Импорты готовы")


## 1. Загрузка данных (download → fallback)

Попробуем загрузить **Telco Customer Churn** из открытого источника. Если не получится — создадим **синтетический** датасет с похожими колонками.

> **Источник данных:** IBM Cognos Analytics sample dataset «Telco Customer Churn» (вымышленная телеком-компания, 7 043 клиента). Файл в репозитории не хранится — он скачивается по ссылкам из кода ниже: [IBM/telco-customer-churn-on-icp4d](https://github.com/IBM/telco-customer-churn-on-icp4d) и зеркало [plotly/datasets](https://github.com/plotly/datasets). Права на данные принадлежат их владельцам.
>
> Если обе ссылки недоступны, ноутбук создаст синтетический датасет, где отток случаен, — тогда метрики во втором ноутбуке будут бессмысленными.


In [ ]:
import pandas as pd

#TO-DO: ипортируйте файл датасета csv-формата с помощью pandas

In [ ]:
# Попытка загрузить реальный датасет (Colab обычно позволяет доступ в интернет).
# Если загрузка не удастся, перейдем к синтетическим данным.
import pandas as pd

urls = [
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv",
]

df = None
for u in urls:
    try:
        df = pd.read_csv(u)
        print("Загружено из:", u)
        break
    except Exception as e:
        print("Не удалось загрузить из:", u, "|", e)

if df is None:
    print("Переходим на синтетический датасет (демо).")
    rng = np.random.default_rng(42)
    n = 500
    df = pd.DataFrame({
        "customerID": [f"C{100000+i}" for i in range(n)],
        "gender": rng.choice(["Male","Female"], size=n),
        "SeniorCitizen": rng.integers(0,2,size=n),
        "Partner": rng.choice(["Yes","No"], size=n, p=[0.45,0.55]),
        "Dependents": rng.choice(["Yes","No"], size=n, p=[0.35,0.65]),
        "tenure": rng.integers(0, 72, size=n),
        "PhoneService": rng.choice(["Yes","No"], size=n, p=[0.9,0.1]),
        "MultipleLines": rng.choice(["Yes","No","No phone service"], size=n),
        "InternetService": rng.choice(["DSL","Fiber optic","No"], size=n, p=[0.4,0.5,0.1]),
        "OnlineSecurity": rng.choice(["Yes","No","No internet service"], size=n),
        "OnlineBackup": rng.choice(["Yes","No","No internet service"], size=n),
        "DeviceProtection": rng.choice(["Yes","No","No internet service"], size=n),
        "TechSupport": rng.choice(["Yes","No","No internet service"], size=n),
        "StreamingTV": rng.choice(["Yes","No","No internet service"], size=n),
        "StreamingMovies": rng.choice(["Yes","No","No internet service"], size=n),
        "Contract": rng.choice(["Month-to-month","One year","Two year"], size=n, p=[0.6,0.2,0.2]),
        "PaperlessBilling": rng.choice(["Yes","No"], size=n, p=[0.7,0.3]),
        "PaymentMethod": rng.choice(["Electronic check","Mailed check","Bank transfer (automatic)","Credit card (automatic)"], size=n),
        "MonthlyCharges": np.round(rng.normal(70, 25, size=n).clip(15, 140), 2),
        "TotalCharges": np.nan,  # заполним ниже как tenure * MonthlyCharges с шумом
        "Churn": rng.choice(["Yes","No"], size=n, p=[0.27,0.73])
    })
    # Сгенерим TotalCharges ~ tenure * MonthlyCharges + noise
    noise = rng.normal(0, 50, size=n)
    df["TotalCharges"] = np.round((df["tenure"] * df["MonthlyCharges"] + noise).clip(0), 2)
    # Внесем немного "грязи": пробелы/строки
    for i in rng.choice(np.arange(n), size=20, replace=False):
        df.loc[i, "TotalCharges"] = " "
    print("Синтетический датасет создан:", df.shape)

print("Первые строки:")
df.head()


## 2. Приведение типов и базовая чистка

В реальном Telco часто встречается проблема: `TotalCharges` приходит как строка с пробелами → нужно привести к числу.


In [ ]:
raw = df.copy()

# Попробуем привести TotalCharges к числу (ошибки -> NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Сколько NaN получили?
print("NaN в TotalCharges:", df["TotalCharges"].isna().sum())

# Заполним NaN нулями для простоты (вариант для демо) — можно выбрать медиану/модельный имьютаор
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Проверим базовые типы
print(df.dtypes.head(10))
df.head(3)


## 3. Быстрый EDA

Посмотрим на распределения и базовые взаимосвязи (по одному графику за раз).


In [ ]:
# Распределение MonthlyCharges
plt.figure(figsize=(6,4))
plt.hist(df["MonthlyCharges"], bins=30)
plt.title("Распределение MonthlyCharges")
plt.xlabel("MonthlyCharges")
plt.ylabel("Count")
plt.show()

In [ ]:
# Распределение TotalCharges
plt.figure(figsize=(6,4))
plt.hist(df["TotalCharges"], bins=30)
plt.title("Распределение TotalCharges")
plt.xlabel("TotalCharges")
plt.ylabel("Count")
plt.show()

In [ ]:
# Tenure vs MonthlyCharges (точечная)
plt.figure(figsize=(6,4))
plt.scatter(df["tenure"], df["MonthlyCharges"])
plt.title("tenure vs MonthlyCharges")
plt.xlabel("tenure (месяцы)")
plt.ylabel("MonthlyCharges")
plt.show()


## 4. Подготовка фич (минимально)

- Бинарные да/нет → 0/1  
- Категориальные → one-hot  
- Цель `Churn` → 0/1

Сохраним подготовленный датасет в файл для следующего ноутбука.


In [ ]:
# Копия для подготовки
prep = df.copy()

# Цель: Churn => 1/0 (Yes/No)
if not pd.api.types.is_numeric_dtype(prep["Churn"]):  # в pandas 3 строки имеют dtype 'str', а не 'object'
    prep["Churn"] = prep["Churn"].map({"Yes":1, "No":0})

# Бинарные "Yes/No" столбцы (если есть)
binary_cols = []
for c in prep.columns:
    if pd.api.types.is_string_dtype(prep[c]):
        # Проверим, что это столбец с Yes/No
        vals = set(prep[c].dropna().unique().tolist())
        if vals.issubset({"Yes","No"}):
            binary_cols.append(c)

for c in binary_cols:
    prep[c] = prep[c].map({"Yes":1, "No":0})

# Категориальные оставшиеся -> one-hot
cat_cols = [c for c in prep.select_dtypes(include=["object", "string"]).columns if c not in ["customerID"]]
prep = pd.get_dummies(prep, columns=cat_cols, drop_first=True)

# Удалим идентификатор
if "customerID" in prep.columns:
    prep = prep.drop(columns=["customerID"])

print("Финальная форма после подготовки:", prep.shape)
prep.head(3)

In [ ]:
# Сохраним подготовленный датасет для второго ноутбука
prep_path = "telco_churn_prepared.csv"
prep.to_csv(prep_path, index=False)
print("Сохранено:", prep_path, "| Размер:", prep.shape)


### Что обсуждаем на занятии
- Какие признаки кажутся «сильными»/«слабыми» и почему?
- Что делать с строковыми пропусками в `TotalCharges` (настоящий кейс из Telco)?
- Какие признаки стоит добавить из бизнес-контекста (например, каналы привлечения)?
- Почему one-hot нужен для категориальных фич?
- Что оставим для модели, а что выбросим и почему?
